# MobileNetV2（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，MobileNetV2を用いてCIFAR100データセットの100クラス物体認識を行う．Depthwise Separable Convolutionを発展させた「Linear Bottleneckを持つInverted Residual構造」を導入することで，スマートフォンなどの計算資源が限られた環境でも高速に動作する軽量なネットワークを構築する方法について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100は32×32ピクセルのカラー画像100クラスからなるデータセットです．学習用データには`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## MobileNetV2
MobileNetV2は，MobileNetV1で導入されたDepthwise Separable Convolution（3×3のDepthwise Convolutionと1×1のPointwise Convolutionを組み合わせることで，通常の畳み込みより少ないパラメータ数・計算量を実現する手法）をベースに，「Linear Bottleneckを持つInverted Residual構造」を導入したネットワークです．

通常のResidual Blockは，チャンネル数の多い特徴マップ同士をショートカット接続で結び，ブロック内部で一度チャンネル数を減らしてから畳み込みを行います（Bottleneck構造）．一方MobileNetV2のInverted Residual Blockは，これと逆に，入力を1×1の畳み込みで一度チャンネル数を拡張（expansion）した上でDepthwise Convolutionを適用し，最後に1×1の畳み込みでチャンネル数を削減（projection）してからショートカット接続します．拡張比率（expand ratio）$t$だけチャンネル数を増やしてからDepthwise Convolutionを行うことで，少ないパラメータ数のまま表現力の高い特徴変換を行うことができます．

さらに，最後のProjection（1×1畳み込み）の直後には活性化関数を適用しません（Linear Bottleneck）．低次元（チャンネル数が少ない）の特徴マップに対してReLUなどの活性化関数を適用すると，非負の値に切り詰められることで情報が失われやすいため，ボトルネック部分では線形変換のままにしておくことで表現力の低下を防いでいます．

なお，ショートカット接続（残差接続）は，入力と出力のチャンネル数が等しく，かつstrideが1の場合のみ行います．

### CIFAR100への入力サイズの適応
MobileNetV2には，224×224ピクセルのImageNet画像を対象とした「ImageNet版」と，32×32ピクセルのCIFAR100のような小さな入力を対象とした「CIFAR版」があります．オリジナルのMobileNetV2はstemとブロック内のstride=2によって特徴マップを合計32分の1まで縮小しますが，32×32の入力にそのまま適用すると特徴マップが1×1以下になってしまいます．そのため，本ノートブックではstemのstrideとブロック内でstride=2となる段数を減らしたCIFAR版を実装します（詳細は本ノートブック末尾の「ImageNet版MobileNetV2とCIFAR版（本ノートブック）の違い」を参照）．

In [ ]:
class InvertedResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride, expand_ratio):
        super().__init__()
        assert stride in (1, 2)
        hidden_ch = in_ch * expand_ratio
        self.use_res = stride == 1 and in_ch == out_ch

        layers = []
        if expand_ratio != 1:
            # 1x1畳み込みでチャンネル数をexpand_ratio倍に拡張
            layers += [nn.Conv2d(in_ch, hidden_ch, kernel_size=1, bias=False),
                       nn.BatchNorm2d(hidden_ch),
                       nn.ReLU6(inplace=True)]
        layers += [
            # 3x3 Depthwise Convolution
            nn.Conv2d(hidden_ch, hidden_ch, kernel_size=3, stride=stride, padding=1, groups=hidden_ch, bias=False),
            nn.BatchNorm2d(hidden_ch),
            nn.ReLU6(inplace=True),
            # 1x1畳み込みでチャンネル数を削減（Projection）．活性化関数は適用しない（Linear Bottleneck）
            nn.Conv2d(hidden_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
        ]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv(x)
        if self.use_res:
            out = out + x
        return out

## ネットワークの定義
MobileNetV2は，Inverted Residual Blockを下の表のように積み重ねて構成します．各行は，拡張比率$t$，出力チャンネル数$c$，ブロックの繰り返し数$n$，各段最初のブロックのstride $s$を表します．

| t | c | n | s |
|---|---|---|---|
| 1 | 16 | 1 | 1 |
| 6 | 24 | 2 | 1 |
| 6 | 32 | 3 | 2 |
| 6 | 64 | 4 | 1 |
| 6 | 96 | 3 | 1 |
| 6 | 160 | 3 | 2 |
| 6 | 320 | 1 | 1 |

最初の3×3畳み込み（stem）で入力画像を32チャンネルに変換したのち，上表のブロックを順に適用します．最後に1×1畳み込みで1280チャンネルに拡張し，Global Average Poolingで空間方向を1×1に集約した後，全結合層でクラス数（100）に変換します．

In [ ]:
def _make_stage(in_ch, out_ch, n_blocks, stride, expand_ratio):
    layers = [InvertedResidual(in_ch, out_ch, stride, expand_ratio)]
    for _ in range(n_blocks - 1):
        layers.append(InvertedResidual(out_ch, out_ch, 1, expand_ratio))
    return nn.Sequential(*layers)


class MobileNetV2(nn.Module):
    # t, c, n, s
    cfg = [
        [1, 16, 1, 1],
        [6, 24, 2, 1],   # 元のImageNet版はs=2だが，CIFAR100 (32x32)向けにs=1へ変更
        [6, 32, 3, 2],
        [6, 64, 4, 1],   # 元のImageNet版はs=2だが，CIFAR100 (32x32)向けにs=1へ変更
        [6, 96, 3, 1],
        [6, 160, 3, 2],
        [6, 320, 1, 1],
    ]

    def __init__(self, n_class=100):
        super().__init__()
        in_ch = 32
        last_ch = 1280

        self.stem = nn.Sequential(
            nn.Conv2d(3, in_ch, kernel_size=3, stride=1, padding=1, bias=False),  # CIFAR100向けにstride=1
            nn.BatchNorm2d(in_ch),
            nn.ReLU6(inplace=True),
        )

        stages = []
        for t, c, n, s in self.cfg:
            stages.append(_make_stage(in_ch, c, n, s, t))
            in_ch = c
        self.blocks = nn.Sequential(*stages)

        self.head = nn.Sequential(
            nn.Conv2d(in_ch, last_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(last_ch),
            nn.ReLU6(inplace=True),
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(last_ch, n_class)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## ネットワークの作成
MobileNetV2を作成し，`.to(device)`によりモデルのパラメータを`device`上に配置します．最適化手法にはモーメンタムSGD（学習率0.01，モーメンタム0.9，weight decay 5e-4）を用います．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
model = MobileNetV2(n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版MobileNetV2とCIFAR版（本ノートブック）の違い

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| 入力画像サイズ | 224×224 | 32×32 |
| stemの畳み込みのstride | 2 | 1 |
| ブロック内でstride=2となる段数 | 4段 | 2段（c=32, c=160の段のみ） |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

MobileNetV2はImageNet（224×224入力）向けに設計されており，stemとブロック内の複数のstride=2によって合計32分の1まで特徴マップを縮小します．CIFAR100は32×32と入力サイズが小さいため，同じ回数だけダウンサンプリングすると特徴マップが1×1以下になってしまいます．そこで本ノートブックでは，stemのstrideを1にし，ブロック内でstride=2となる段数を4段から2段に減らすことで，最終的な特徴マップサイズを8×8（4分の1への縮小）としています．

## 課題

1. 拡張比率（expand ratio）$t$の値を変更して，パラメータ数や認識精度がどのように変化するか確認しましょう．
2. Linear Bottleneck（Projection直後で活性化関数を適用しない構造）を取り除き，通常のReLU6を適用した場合とで学習の推移や認識精度がどのように異なるか比較しましょう．
3. ショートカット接続（残差接続）を取り除いたネットワークを作成し，Inverted Residual構造の効果を確認しましょう．